In [2]:
import json, re, ast, math
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional

SRC = Path("results_monthly_generic_20250821_150909.jsonl")
OUT_DIR = Path("parsed_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 1) JSON 보정 유틸 ----------
TRAILING_COMMA_RE = re.compile(r",\s*([}\]])")
DANGLING_COMMA_LINE_RE = re.compile(r",\s*$", re.M)

def extract_largest_object(text: str) -> Optional[str]:
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return text[start:]  # 닫힘이 없다면 끝까지

def fix_json_like(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = "".join(ch for ch in s if ord(ch) >= 9)  # 제어문자 제거
    s = re.sub(r"//.*?$", "", s, flags=re.M)     # // 주석 제거
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.S)  # /* */ 주석 제거
    maybe = extract_largest_object(s) or s
    maybe = TRAILING_COMMA_RE.sub(r"\1", maybe)      # }], 직전 콤마 제거
    maybe = DANGLING_COMMA_LINE_RE.sub("", maybe)    # 줄 끝 콤마 제거

    # 문자열 밖의 파이썬 리터럴 -> JSON 리터럴
    def replace_python_literals(text: str) -> str:
        res, in_str, quote, i = [], False, None, 0
        while i < len(text):
            ch = text[i]
            if not in_str and ch in ('"', "'"):
                in_str, quote = True, ch
                res.append(ch); i += 1
            elif in_str:
                res.append(ch)
                if ch == "\\" and i+1 < len(text):
                    res.append(text[i+1]); i += 2; continue
                if ch == quote:
                    in_str, quote = False, None
                i += 1
            else:
                if text.startswith("None", i):
                    res.append("null"); i += 4
                elif text.startswith("True", i):
                    res.append("true"); i += 4
                elif text.startswith("False", i):
                    res.append("false"); i += 5
                else:
                    res.append(ch); i += 1
        return "".join(res)
    maybe = replace_python_literals(maybe)

    # 값이 '...' 인 경우만 "..."로 치환 (키는 이미 "key"라고 가정)
    maybe = re.sub(
        r':\s*\'([^\']*)\'',
        lambda m: ': "' + m.group(1).replace('"', '\\"') + '"',
        maybe
    )
    return maybe

def balance_braces(text: str) -> str:
    stack, pairs = [], {']':'[', '}':'{'}
    openers, closers = set(pairs.values()), set(pairs.keys())
    for ch in text:
        if ch in openers: stack.append(ch)
        elif ch in closers and stack and stack[-1] == pairs[ch]: stack.pop()
    close_for = {'{': '}', '[': ']'}
    return text + "".join(close_for[c] for c in reversed(stack))

def try_parse_json_v2(raw: str) -> Optional[Dict[str, Any]]:
    if not raw:
        return None
    try:
        return json.loads(raw)
    except Exception:
        pass
    largest = extract_largest_object(raw) or raw
    largest = fix_json_like(largest)
    largest = balance_braces(largest)
    try:
        return json.loads(largest)
    except Exception:
        try:
            obj = ast.literal_eval(largest)
            if isinstance(obj, dict):
                return obj
        except Exception:
            return None
    return None

# ---------- 2) 느슨한(정규식) 추출 ----------
product_pattern = re.compile(
    r'\{[^{}]*"product"\s*:\s*"(?P<product>[^"]+)"[^{}]*'
    r'"persona_fit_score"\s*:\s*(?P<fit>[-+]?\d*\.?\d+)[^{}]*'
    r'"purchase_prob"\s*:\s*(?P<prob>[-+]?\d*\.?\d+)[^{}]*'
    r'"expected_qty"\s*:\s*(?P<qty>[-+]?\d*\.?\d+|\d+)[^{}]*'
    r'"expected_spend_KRW"\s*:\s*(?P<spend>[-+]?\d*\.?\d+|\d+)',
    re.S
)
pid_pattern = re.compile(r'"persona_id"\s*:\s*(\d+)', re.S)

rows, parsed_lines, total_lines = [], 0, 0
with open(SRC, "r", encoding="utf-8") as f:
    for line in f:
        total_lines += 1
        try:
            rec = json.loads(line)
        except Exception:
            continue
        text = rec.get("raw_text", "") or ""
        pid = rec.get("persona_id")
        m_pid = pid_pattern.search(text)
        if m_pid: pid = int(m_pid.group(1))

        # 1) 보정 파서 먼저 시도 (성공 시 정석 경로)
        obj = try_parse_json_v2(text)
        if obj and isinstance(obj, dict) and isinstance(obj.get("forecast"), list):
            for it in obj["forecast"]:
                if not isinstance(it, dict): continue
                rows.append({
                    "persona_id": obj.get("persona_id", pid),
                    "product": it.get("product"),
                    "persona_fit_score": it.get("persona_fit_score"),
                    "purchase_prob": it.get("purchase_prob"),
                    "expected_qty": it.get("expected_qty"),
                    "expected_spend_KRW": it.get("expected_spend_KRW"),
                })
            parsed_lines += 1
            continue

        # 2) 보정 실패하면 정규식으로 제품 블록만 추출
        found_any = False
        for m in product_pattern.finditer(text):
            found_any = True
            rows.append({
                "persona_id": pid,
                "product": m.group("product"),
                "persona_fit_score": float(m.group("fit")),
                "purchase_prob": float(m.group("prob")),
                "expected_qty": float(m.group("qty")),
                "expected_spend_KRW": float(m.group("spend")),
            })
        if found_any:
            parsed_lines += 1

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "forecast_rows_loose.csv", index=False, encoding="utf-8-sig")

summary = (
    df.assign(
        persona_fit_score=pd.to_numeric(df["persona_fit_score"], errors="coerce"),
        purchase_prob=pd.to_numeric(df["purchase_prob"], errors="coerce"),
        expected_qty=pd.to_numeric(df["expected_qty"], errors="coerce"),
        expected_spend_KRW=pd.to_numeric(df["expected_spend_KRW"], errors="coerce"),
    )
    .groupby("product")
    .agg(
        personas=("persona_id", "nunique"),
        avg_fit=("persona_fit_score", "mean"),
        avg_prob=("purchase_prob", "mean"),
        total_qty=("expected_qty", "sum"),
        total_spend_KRW=("expected_spend_KRW", "sum"),
    )
    .reset_index()
    .sort_values(["total_spend_KRW", "total_qty", "avg_prob"], ascending=[False, False, False])
)
summary.to_csv(OUT_DIR / "forecast_summary_by_product_loose.csv", index=False, encoding="utf-8-sig")

with open(OUT_DIR / "parse_stats.txt", "w", encoding="utf-8") as f:
    f.write(f"Total lines: {total_lines}\n")
    f.write(f"Lines with parsed content: {parsed_lines}\n")
    f.write(f"Total extracted rows: {len(df)}\n")


In [7]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Optional

FORECAST_ROWS = Path("parsed_results/forecast_rows_loose.csv")
PERSONA_PATHS = [
    Path("persona.json"),    # 또는 전체가 list/dict 인 JSON
]

# 1) 페르소나 로더 (JSONL/JSON 모두 지원) + 평탄화
def load_personas(paths: List[Path]) -> pd.DataFrame:
    for p in paths:
        if p.exists():
            if p.suffix.lower() == ".jsonl":
                rows = []
                with open(p, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            rows.append(json.loads(line))
                        except Exception:
                            continue
                meta = pd.json_normalize(rows, sep=".") if rows else pd.DataFrame()
            else:
                obj = json.loads(p.read_text(encoding="utf-8"))
                if isinstance(obj, list):
                    meta = pd.json_normalize(obj, sep=".")
                elif isinstance(obj, dict):
                    # dict이면 value가 list일 가능성 처리
                    if "personas" in obj and isinstance(obj["personas"], list):
                        meta = pd.json_normalize(obj["personas"], sep=".")
                    else:
                        meta = pd.json_normalize([obj], sep=".")
                else:
                    meta = pd.DataFrame()

            if not meta.empty:
                return meta
    # 못 찾으면 빈 DF
    return pd.DataFrame()

def to_age_group(age: Optional[float]) -> str:
    try:
        a = int(age)
    except Exception:
        return "기타"
    if a < 20:   return "10대-"
    if a < 30:   return "20대"
    if a < 40:   return "30대"
    if a < 50:   return "40대"
    if a < 60:   return "50대"
    if a < 70:   return "60대"
    return "70대+"

# 2) 데이터 로드
df_forecast = pd.read_csv(FORECAST_ROWS)
persona_meta = load_personas(PERSONA_PATHS)

# 3) 필요한 컬럼만 평탄화 (demographics.age, demographics.gender)
#    예: "demographics.age", "demographics.gender" 처럼 점표기 열이 존재해야 함
if "demographics.age" not in persona_meta.columns and "age" in persona_meta.columns:
    # 사용자가 이미 평탄화된 파일을 쓰는 경우 호환
    persona_meta["demographics.age"] = persona_meta["age"]
if "demographics.gender" not in persona_meta.columns and "gender" in persona_meta.columns:
    persona_meta["demographics.gender"] = persona_meta["gender"]

need_cols = ["persona_id", "demographics.age", "demographics.gender"]
for c in need_cols:
    if c not in persona_meta.columns:
        raise KeyError(f"페르소나 메타데이터에 '{c}' 컬럼이 없습니다. (파일 평탄화/경로 확인)")

meta = persona_meta[need_cols].copy()
meta["age_group"] = meta["demographics.age"].apply(to_age_group)

# 성별 표준화 (남자/여자로 통일)
def norm_gender(g):
    s = str(g or "").strip()
    if s in ["남", "남성", "male", "MALE", "M"]: return "남자"
    if s in ["여", "여성", "female", "FEMALE", "F"]: return "여자"
    return s if s in ["남자","여자"] else "기타"
meta["gender"] = meta["demographics.gender"].apply(norm_gender)

# 4) 목표 인구분포 정의 (★여기를 실제 분포로 바꿔주세요)
target_dist = pd.DataFrame({
    "age_group": ["20대","30대","40대","50대","60대","70대+"],
    "남자":      [0.08,  0.09,  0.09,  0.08,  0.06,  0.05],
    "여자":      [0.08,  0.09,  0.10,  0.09,  0.07,  0.05],
})
target_long = target_dist.melt(id_vars="age_group", var_name="gender", value_name="target_share")

# 5) 샘플(페르소나) 분포 계산
sample_dist = (
    meta.groupby(["age_group","gender"])["persona_id"]
        .count()
        .reset_index(name="count")
)
sample_total = sample_dist["count"].sum()
sample_dist["sample_share"] = sample_dist["count"] / (sample_total if sample_total else 1)

# 6) weight = target_share / sample_share
weights = pd.merge(sample_dist, target_long, on=["age_group","gender"], how="outer").fillna(0.0)
# 샘플에 없는 셀은 sample_share=0이므로 weight 무한 방지
import numpy as np
weights["weight"] = np.where(weights["sample_share"]>0,
                             weights["target_share"]/weights["sample_share"],
                             0.0)

# 7) 예측 데이터와 가중치 결합
use_cols = ["persona_id","age_group","gender"]
dfw = pd.merge(
    df_forecast,
    meta[use_cols],
    on="persona_id",
    how="left"
)
dfw = pd.merge(
    dfw,
    weights[["age_group","gender","weight"]],
    on=["age_group","gender"],
    how="left"
)
dfw["weight"] = dfw["weight"].fillna(0.0)

# 수치형 변환
for col in ["persona_fit_score","purchase_prob","expected_qty","expected_spend_KRW"]:
    dfw[col] = pd.to_numeric(dfw[col], errors="coerce")

# 8) 가중 합산 요약
summary_weighted = (
    dfw.groupby("product")
       .apply(lambda g: pd.Series({
           "personas": g["persona_id"].nunique(),
           "avg_fit_w": (g["persona_fit_score"]*g["weight"]).sum() / (g["weight"].sum() or 1),
           "avg_prob_w": (g["purchase_prob"]*g["weight"]).sum() / (g["weight"].sum() or 1),
           "total_qty_w": (g["expected_qty"]*g["weight"]).sum(),
           "total_spend_KRW_w": (g["expected_spend_KRW"]*g["weight"]).sum(),
       }))
       .reset_index()
       .sort_values(["total_spend_KRW_w","total_qty_w","avg_prob_w"], ascending=[False, False, False])
)

OUT = Path("parsed_results/forecast_summary_by_product_WEIGHTED.csv")
summary_weighted.to_csv(OUT, index=False, encoding="utf-8-sig")
print("✅ Weighted summary saved:", OUT)
print(summary_weighted.head(10))


✅ Weighted summary saved: parsed_results\forecast_summary_by_product_WEIGHTED.csv
            product  personas  avg_fit_w  avg_prob_w  total_qty_w  \
0  덴마크 하이그릭요거트 400g    2140.0   0.669188    0.294819  3322.130401   
6      동원참치액 순 900g    2140.0   0.566358    0.313299  2300.480467   
8      동원참치액 진 900g    2140.0   0.502008    0.256608  1957.029173   
5      동원참치액 순 500g    2140.0   0.642691    0.385767  3075.820325   
7      동원참치액 진 500g    2140.0   0.575121    0.324740  2392.081308   
9      리챔 오믈레햄 200g    2127.0   0.588192    0.329760  2737.216814   
1   동원맛참 고소참기름 135g    2140.0   0.601822    0.358563  3053.373118   
3   동원맛참 매콤참기름 135g    2140.0   0.527148    0.257019  2267.261088   
2    동원맛참 고소참기름 90g    2140.0   0.517421    0.283951  2409.066691   
4    동원맛참 매콤참기름 90g    2140.0   0.453633    0.202933  1902.321804   

   total_spend_KRW_w  
0      738873.993599  
6      551897.037523  
8      522390.870380  
5      503582.689094  
7      455233.417634  
9      439520.752625

C:\Users\Admin\AppData\Local\Temp\ipykernel_2240\3289132446.py:134: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
